In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


In [4]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img

In [5]:
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    directory='/content/brain-tumor-mri-dataset/Training',
    target_size=(128,128),
    batch_size=32,
    class_mode='sparse'



)

test_generator = datagen.flow_from_directory(
    directory='/content/brain-tumor-mri-dataset/Testing',
    target_size=(128,128),
    batch_size=32,
    class_mode='sparse'

)

Found 5600 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.


In [6]:
model = keras.Sequential([

    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),

    keras.layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_generator,validation_data=test_generator,epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 23s 81ms/step - accuracy: 0.7368 - loss: 0.7367 - val_accuracy: 0.3038 - val_loss: 4.0442
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 73ms/step - accuracy: 0.8305 - loss: 0.4292 - val_accuracy: 0.3894 - val_loss: 2.2336
Epoch 3/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - accuracy: 0.8711 - loss: 0.3253 - val_accuracy: 0.7837 - val_loss: 0.7186
Epoch 4/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 75ms/step - accuracy: 0.9089 - loss: 0.2369 - val_accuracy: 0.8544 - val_loss: 0.6594
Epoch 5/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - accuracy: 0.9261 - loss: 0.1970 - val_accuracy: 0.8625 - val_loss: 0.8656
Epoch 6/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 73ms/step - accuracy: 0.9448 - loss: 0.1432 - val_accuracy: 0.8794 - val_loss: 0.7431
Epoch 7/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 71ms/step - accuracy: 0.9591 - loss: 0.1126 - val_accuracy: 0.8906 - val_loss: 0.6891
Epoch 8/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - accuracy: 0.9618 - loss: 0.0996 - 

In [7]:
model.evaluate(test_generator)

50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.8919 - loss: 0.8833


[0.8833296895027161, 0.8918750286102295]

In [8]:
from sklearn.metrics import classification_report
import numpy as np

y_pred = model.predict(test_generator)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(test_generator.classes, y_pred))

50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step
              precision    recall  f1-score   support

           0       0.24      0.18      0.21       400
           1       0.22      0.26      0.24       400
           2       0.24      0.28      0.25       400
           3       0.26      0.26      0.26       400

    accuracy                           0.24      1600
   macro avg       0.24      0.24      0.24      1600
weighted avg       0.24      0.24      0.24      1600



In [9]:
model.save('brain_tumor_mri.keras')